In [1]:
from flask import Flask,jsonify,request
import tensorflow as tf
import numpy as np

In [2]:
model = tf.keras.models.load_model('Mymodel.h5')

In [3]:
app=Flask(__name__)

In [4]:
def names(number):
    if number == 0:
        return 'Non Demented'
    elif number == 1:
        return 'Mild Dementia'
    elif number == 2:
        return 'Moderate Dementia'
    elif number == 3:
        return 'Very Mild Dementia'
    else:
        return 'Error in Prediction'
# 0 --> Non Demented
# 1 --> Mild Dementia
# 2 --> Moderate Dementia
# 3 --> Very Mild Dementia 

In [5]:
import base64
import io
from PIL import Image
import numpy as np

def decode_base64_image(base64_string):
    # Remove the header (if present)
    if base64_string.startswith('data:image'):
        base64_string = base64_string.split(',')[1]
    
    try:
        image_data = base64.b64decode(base64_string)
        image = Image.open(io.BytesIO(image_data))
        return image
    except Exception as e:
        print(f"Error decoding base64 image: {e}")
        return None

In [6]:
@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()
        img_base64 = data.get("imageData")
        
        if not img_base64:
            return jsonify({"error": "No image data provided"}), 400
        
        image = decode_base64_image(img_base64)
        if image is None:
            return jsonify({"error": "Failed to decode image"}), 400
        
        image_resized = image.resize((128, 128))
        image_array = np.array(image_resized)
        input_data = image_array.reshape(1, 128, 128, 3)
        prediction = model.predict_on_batch(input_data)
        classification = np.argmax(prediction)
        response = f"{prediction[0][classification] * 100:.2f}% Confidence This Is {names(classification)}"
        print(response)
        return jsonify({"Results": response})
    
    except Exception as e:
        print(f"Error during prediction: {e}")
        return jsonify({"error": str(e)}), 500


In [7]:
if __name__=='__main__':
    app.run(debug=True , use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Jul/2024 15:47:25] "POST /predict HTTP/1.1" 200 -


100.00% Confidence This Is Mild Dementia


127.0.0.1 - - [03/Jul/2024 15:47:40] "POST /predict HTTP/1.1" 200 -


100.00% Confidence This Is Moderate Dementia


...

100.00% Confidence This Is Mild Dementia


127.0.0.1 - - [03/Jul/2024 15:47:50] "POST /predict HTTP/1.1" 200 -


100.00% Confidence This Is Very Mild Dementia


127.0.0.1 - - [03/Jul/2024 15:47:58] "POST /predict HTTP/1.1" 200 -


100.00% Confidence This Is Very Mild Dementia
